# 1. Configuration

Every world and system in TidalPy is described by a small **TOML** configuration file. This first demo
shows where those files live, what they look like, and how to load one, read it as a plain Python
dictionary, edit it in a live session, and build a world from it.

We only cover the configuration system here. Building and using the worlds themselves comes next, in
`02_world_building`.

*Figures use `%matplotlib inline`; there are none in this notebook. Switch any later notebook's first
cell to `%matplotlib widget` (needs `ipympl`) for interactive pan and zoom.*


## Where configurations live

TidalPy ships a small pack of example worlds. `available_worlds()` lists the ones it can find by name, and `get_worlds_x_dir()` is the per-user directory they are installed into. Every config also carries a `schema_version` so TidalPy knows how to read it.

In [1]:
from TidalPy.structures_x.configs import (
    available_worlds,
    get_worlds_x_dir,
    resolve_world_path,
    load_toml,
    merge_with_defaults,
    validate_world_config,
    build_world,
    SCHEMA_VERSION,
)

print("current schema version:", SCHEMA_VERSION)
print("worlds directory      :", get_worlds_x_dir())
print("worlds available      :", available_worlds())


current schema version: 0.2.0
worlds directory      : C:\Users\joepr\Documents\TidalPy\0.8.X\Worlds_x
worlds available      : ['earth_prem', 'earth_simple', 'jupiter_simple', 'sol', 'sol_system']


## What a config file looks like

`resolve_world_path` turns a world name into the path of its TOML file. Here is the raw text of the two-layer `earth_simple` world: a name, bulk properties (radius, mass, spin), and one table per layer. Each layer names a `class` (which layer type to build) and a material `type`; the detailed material defaults (equation of state, rheology, viscosity, and so on) are filled in from TidalPy's material library at build time.

In [2]:
from pathlib import Path

earth_path = resolve_world_path("earth_simple")
print(earth_path)
print()
print(Path(earth_path).read_text())


C:\Users\joepr\Documents\TidalPy\0.8.X\Worlds_x\earth_simple.toml

# Simple two-layer terrestrial world for the structures_x world builder.
# A non-tidal iron core under a tidally active rocky mantle. Each layer names a
# `class` (which layer class to build) and a material `type`; the per-material
# parameter defaults (EOS, rheology, viscosity, melt, cooling, radiogenics) are
# pulled from the matching [layers.<type>] block of TidalPy_Configs_x.toml. Any of
# those can be overridden here by adding the corresponding key or [..] sub-table.
#
# Layers are built inner-to-outer: the inner radius is derived from the previous
# layer (0 for the innermost), and each layer's outer radius is set by exactly one of
# radius_outer_m, radius_fraction (* world radius), or volume_fraction (* world volume).
schema_version = "0.2.0"
name = "Earth-Simple"
type = "terrestrial"
radius_m = 6371000.0
mass_kg = 5.972e24
spin_frequency_rad_s = 7.292e-5

[layers.core]
class = "physics"
type = "iron"
layer_index

## Loading a config as a dictionary

`load_toml` reads a TOML file (by path) into an ordinary nested dictionary. This is the form you edit in a live session.

In [3]:
import pprint

config = load_toml(earth_path)
pprint.pp(config)


{'schema_version': '0.2.0',
 'name': 'Earth-Simple',
 'type': 'terrestrial',
 'radius_m': 6371000.0,
 'mass_kg': 5.972e+24,
 'spin_frequency_rad_s': 7.292e-05,
 'layers': {'core': {'class': 'physics',
                     'type': 'iron',
                     'layer_index': 0,
                     'radius_outer_m': 3480000.0,
                     'is_tidal': False},
            'mantle': {'class': 'solidliquid',
                       'type': 'mantle_rock',
                       'layer_index': 1,
                       'radius_fraction': 1.0,
                       'is_tidal': True}}}


## Building a world from a name or a dictionary

`build_world` accepts either a world name (which it resolves and loads for you) or a config dictionary. Both give back the same kind of world object.

In [4]:
world_from_name = build_world("earth_simple")
world_from_dict = build_world(config)

for w in (world_from_name, world_from_dict):
    print(f"{w.name:14} type={w.world_type:12} R={w.radius:.3e} m  M={w.mass:.3e} kg  layers={w.num_layers}")


Earth-Simple   type=terrestrial  R=6.371e+06 m  M=5.972e+24 kg  layers=2
Earth-Simple   type=terrestrial  R=6.371e+06 m  M=5.972e+24 kg  layers=2


## Editing a config in a live session

Because the config is just a dictionary, you can change it in memory and build a new world from the result, without touching any file on disk. Here we make a heavier planet with a larger iron core. `validate_world_config` checks the edited dictionary against the schema before we build, so mistakes are caught early.

In [5]:
import copy

edited = copy.deepcopy(config)
edited["name"] = "Earth-Heavy"
edited["mass_kg"] = 7.0e24                       # heavier world
edited["layers"]["core"]["radius_outer_m"] = 4.0e6   # larger core

validate_world_config(edited)   # raises if the edit broke the schema
heavy = build_world(edited)

print(f"{'world':14} {'mass (kg)':>12} {'core R (m)':>12} {'mean rho (kg/m^3)':>18}")
for w, cfg in ((world_from_name, config), (heavy, edited)):
    print(f"{w.name:14} {w.mass:12.3e} {cfg['layers']['core']['radius_outer_m']:12.3e} {w.calc_mean_density():18.1f}")


world             mass (kg)   core R (m)  mean rho (kg/m^3)
Earth-Simple      5.972e+24    3.480e+06             5513.3
Earth-Heavy       7.000e+24    4.000e+06             6462.3


## Optional: applying material defaults yourself

`build_world` merges the compact config with TidalPy's material defaults for you. If you want to see or adjust that merged form directly, call `merge_with_defaults`.

In [6]:
merged = merge_with_defaults(config)
print("top-level keys after merge:", list(merged.keys()))
print("mantle layer:", merged["layers"]["mantle"])


top-level keys after merge: ['schema_version', 'name', 'type', 'radius_m', 'mass_kg', 'spin_frequency_rad_s', 'layers']
mantle layer: {'class': 'solidliquid', 'type': 'mantle_rock', 'layer_index': 1, 'radius_fraction': 1.0, 'is_tidal': True}


## Recap

- Worlds and systems are described by TOML config files carrying a `schema_version`.
- `available_worlds()`, `get_worlds_x_dir()`, and `resolve_world_path()` find the bundled examples.
- `load_toml()` reads a file into a dictionary; `merge_with_defaults()` fills in material defaults;
  `validate_world_config()` checks it.
- `build_world()` takes either a name or a config dictionary, so you can edit a config live and rebuild.

Next: **`02_world_building`** builds the different world types from these configs and runs basic
calculations on them.
